
## Módulo 2 - Motor de Alertas Tempranas

Este notebook convierte los hallazgos del Módulo 1 en un motor reproducible de alertas operacionales.

**Flujo:**

```text
Forecast climático
      ↓
Zona + hora + precipitación
      ↓
Ratio proyectado
      ↓
Nivel de riesgo
      ↓
Earnings recomendado
      ↓
Deduplicación
      ↓
Alerta operacional
```



In [ ]:
# 0 — Instalación y carga de datos

!pip -q install pandas openpyxl statsmodels numpy scipy shapely requests

import requests
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from shapely import wkt
from google.colab import files

uploaded = files.upload()
excel_files = [x for x in uploaded.keys() if x.lower().endswith((".xlsx", ".xls"))]

if not excel_files:
    raise FileNotFoundError("No se encontró el Excel del caso.")

PATH = excel_files[0]

raw = pd.read_excel(PATH, sheet_name="RAW_DATA")
zone_info = pd.read_excel(PATH, sheet_name="ZONE_INFO")

df = raw.copy()
df["DATE"] = pd.to_datetime(df["DATE"])
df["RATIO"] = df["ORDERS"] / df["CONNECTED_RT"]
df["SATURATION"] = df["RATIO"] > 1.8

# Sensibilidad histórica por zona, utilizada posteriormente por el motor.
zone_sensitivity = []

for zone, group in df.groupby("ZONE"):
    model = smf.ols(
        "RATIO ~ PRECIPITATION_MM + C(HOUR)",
        data=group
    ).fit()

    zone_sensitivity.append({
        "ZONE": zone,
        "rain_slope": model.params.get("PRECIPITATION_MM", np.nan),
        "p_value": model.pvalues.get("PRECIPITATION_MM", np.nan),
        "r_squared": model.rsquared
    })

zone_sensitivity = (
    pd.DataFrame(zone_sensitivity)
      .sort_values("rain_slope", ascending=False)
)

print(f"Archivo cargado: {PATH}")
print(f"Filas: {len(df):,}")
print(f"Zonas históricas: {df['ZONE'].nunique()}")


## Decisiones heredadas del diagnóstico histórico

El Módulo 1 identificó:

- **12–14 hrs** como ventana de mayor riesgo.
- **>1 mm/hr** como trigger climático preventivo inicial en la ventana crítica.
- Diferencias de sensibilidad entre zonas.
- La necesidad de separar **estimación de riesgo** de **calibración de earnings**.

Estas decisiones se utilizan como hipótesis operacionales y no como relaciones causales demostradas.


## 2A. Forecast de precipitación + mapeo por zonas

Elegí **Open-Meteo** porque el propio caso lo recomienda: no requiere API key y entrega precipitación horaria.

La documentación actual de Open-Meteo permite enviar múltiples coordenadas y devolver un forecast horario para cada ubicación.

### ¿Cómo use los polígonos?

1. Leer `ZONE_POLYGONS`.
2. Convertir WKT → polígonos Shapely.
3. Tomar un punto representativo dentro de cada polígono.
4. Consultar el forecast para esas coordenadas.
5. Validar que el punto retornado por la API cae en la zona esperada.


In [ ]:
# 2A. Cargar polígonos y preparar coordenadas para Open-Meteo

!pip -q install shapely requests

import requests
import pandas as pd
import numpy as np
from shapely import wkt
from shapely.geometry import Point
from datetime import datetime, timedelta

zone_polygons = pd.read_excel(PATH, sheet_name="ZONE_POLYGONS")
zone_info = pd.read_excel(PATH, sheet_name="ZONE_INFO")

# Diccionario de centros disponibles en ZONE_INFO
centers = zone_info.set_index("ZONE")[["LATITUDE_CENTER", "LONGITUDE_CENTER"]].to_dict("index")

def safe_load_wkt(value):
    try:
        geom = wkt.loads(value)
        if geom.is_valid:
            return geom, "polygon"
        # Si la geometría es válida pero tiene problemas topológicos
        return geom.buffer(0), "polygon_repaired"
    except Exception:
        return None, "fallback_center"

records = []

for _, row in zone_polygons.iterrows():
    geom, method = safe_load_wkt(row["GEOMETRY_WKT"])
    zone = row["ZONE_NAME"]

    if geom is not None:
        p = geom.representative_point()

        # Validación point-in-polygon: el punto usado para el forecast
        # debe pertenecer a la geometría de la zona.
        if not geom.contains(p):
            raise ValueError(f"El punto representativo no pertenece a {zone}")

        lat, lon = p.y, p.x
    else:
        lat = centers[zone]["LATITUDE_CENTER"]
        lon = centers[zone]["LONGITUDE_CENTER"]

    records.append({
        "ZONE_ID": row["ZONE_ID"],
        "ZONE": zone,
        "LAT_QUERY": lat,
        "LON_QUERY": lon,
        "MAPPING_METHOD": method
    })

zone_points = pd.DataFrame(records)

display(zone_points)

print("\nMétodo de mapeo:")
print(zone_points["MAPPING_METHOD"].value_counts())


,ZONE_ID,ZONE,LAT_QUERY,LON_QUERY,MAPPING_METHOD
0,919,Centro,25.683833,-100.321905,polygon
1,1371,Mitras Centro,25.721842,-100.389035,polygon
2,901,Apodaca Centro,25.789859,-100.196235,polygon
3,1372,Escobedo,25.808297,-100.306611,polygon
4,944,Carretera Nacional,25.548700,-100.232000,fallback_center
5,1369,MTY_Apodaca_Huinalá,25.709077,-100.152515,polygon
6,928,San Nicolás,25.740678,-100.289930,polygon
7,922,Santa Catarina,25.689585,-100.461292,polygon
8,939,San Pedro,25.642608,-100.364042,polygon
9,908,Cumbres Poniente,25.781051,-100.416416,polygon



Método de mapeo:
MAPPING_METHOD
polygon            12
fallback_center     2
Name: count, dtype: int64



La columna `MAPPING_METHOD` es importante:

- `polygon` → usar el polígono real.
- `polygon_repaired` → corregir una geometría topológica sin cambiar su área de forma significativa.
- `fallback_center` → el WKT estaba truncado y use el centro disponible en `ZONE_INFO`.

### ¿Qué devuelve el forecast?

Para cada zona tendremos algo parecido a:

```text
ZONE              HOUR       PRECIPITATION_MM
Santiago          12:00      7.2
Santiago          13:00      6.8
Santiago          14:00      5.9
...
```


In [ ]:
# 2A. Consultar forecast horario para las 14 zonas

def get_open_meteo_forecast(zone_points, forecast_days=2):
    lats = ",".join(zone_points["LAT_QUERY"].round(6).astype(str))
    lons = ",".join(zone_points["LON_QUERY"].round(6).astype(str))

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": lats,
        "longitude": lons,
        "hourly": "precipitation",
        "forecast_days": forecast_days,
        "timezone": "America/Monterrey"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    payload = response.json()

    # Open-Meteo devuelve una lista cuando se solicitan múltiples coordenadas.
    if not isinstance(payload, list):
        payload = [payload]

    rows = []

    for i, item in enumerate(payload):
        zone_row = zone_points.iloc[i]

        for timestamp, precipitation in zip(
            item["hourly"]["time"],
            item["hourly"]["precipitation"]
        ):
            rows.append({
                "ZONE": zone_row["ZONE"],
                "LAT_QUERY": zone_row["LAT_QUERY"],
                "LON_QUERY": zone_row["LON_QUERY"],
                "FORECAST_TIME": pd.to_datetime(timestamp),
                "PRECIPITATION_FORECAST_MM": precipitation,
                "MAPPING_METHOD": zone_row["MAPPING_METHOD"],
                "API_LATITUDE": item.get("latitude"),
                "API_LONGITUDE": item.get("longitude")
            })

    return pd.DataFrame(rows)

forecast = get_open_meteo_forecast(zone_points)

print("Forecast rows:", len(forecast))
print("Zones:", forecast["ZONE"].nunique())
display(forecast.head(20))


Forecast rows: 672
Zones: 14


,ZONE,LAT_QUERY,LON_QUERY,FORECAST_TIME,PRECIPITATION_FORECAST_MM,MAPPING_METHOD,API_LATITUDE,API_LONGITUDE
0,Centro,25.683833,-100.321905,2026-08-31 00:00:00,0.0,polygon,25.692347,-100.31243
1,Centro,25.683833,-100.321905,2026-08-31 01:00:00,0.0,polygon,25.692347,-100.31243
2,Centro,25.683833,-100.321905,2026-08-31 02:00:00,0.0,polygon,25.692347,-100.31243
3,Centro,25.683833,-100.321905,2026-08-31 03:00:00,0.0,polygon,25.692347,-100.31243
4,Centro,25.683833,-100.321905,2026-08-31 04:00:00,0.0,polygon,25.692347,-100.31243
5,Centro,25.683833,-100.321905,2026-08-31 05:00:00,0.0,polygon,25.692347,-100.31243
6,Centro,25.683833,-100.321905,2026-08-31 06:00:00,0.0,polygon,25.692347,-100.31243
7,Centro,25.683833,-100.321905,2026-08-31 07:00:00,0.0,polygon,25.692347,-100.31243
8,Centro,25.683833,-100.321905,2026-08-31 08:00:00,0.0,polygon,25.692347,-100.31243
9,Centro,25.683833,-100.321905,2026-08-31 09:00:00,0.0,polygon,25.692347,-100.31243



Para el MVP utilizare **2 horas**:

- 1h → más preciso, pero poco tiempo para reaccionar.
- 3h → más tiempo de reacción, pero mayor incertidumbre.
- **2h → punto intermedio** y además coincide con el formato de alerta solicitado


# 2B. Ratio proyectado

Es necesario convertir:

**lluvia forecast + hora + zona**

en:

**ratio proyectado**

### Modelo histórico

Usare:

```text
RATIO ~ PRECIPITATION_MM + HOUR + ZONE
```

**NO incluye earnings aquí.** Porque primero es necesario estimar el riesgo operacional que existiría bajo las condiciones climáticas previstas, antes de decidir cuánto pagar.



In [ ]:
# 2B. Modelo de ratio proyectado

ratio_model = smf.ols(
    "RATIO ~ PRECIPITATION_MM + C(HOUR) + C(ZONE)",
    data=df
).fit()

print(ratio_model.summary().tables[1])
print("\nR² del modelo:", round(ratio_model.rsquared, 3))


                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                          0.2217      0.018     12.469      0.000       0.187       0.257
C(HOUR)[T.1]                       0.2909      0.020     14.368      0.000       0.251       0.331
C(HOUR)[T.2]                       0.2904      0.020     14.342      0.000       0.251       0.330
C(HOUR)[T.3]                       0.2713      0.020     13.399      0.000       0.232       0.311
C(HOUR)[T.4]                       0.3003      0.020     14.830      0.000       0.261       0.340
C(HOUR)[T.5]                       0.2929      0.020     14.467      0.000       0.253       0.333
C(HOUR)[T.6]                       0.2921      0.020     14.427      0.000       0.252       0.332
C(HOUR)[T.7]                      -0.0616      0.020     -3.040      0.002      -0.101      -0.022
C(HOUR)[T.

In [ ]:
# 2B. Función para proyectar ratio

def project_ratio(zone, hour, precipitation):
    x = pd.DataFrame({
        "PRECIPITATION_MM": [precipitation],
        "HOUR": [int(hour)],
        "ZONE": [zone]
    })

    return float(ratio_model.predict(x).iloc[0])

# Ejemplo del caso:
example_ratio = project_ratio(
    zone="Santiago",
    hour=14,
    precipitation=7.2
)

print(f"Ratio proyectado: {example_ratio:.2f}")


Ratio proyectado: 2.01




Este modelo responde:

**Si históricamente esta zona y esta hora presentan esta cantidad de lluvia, ¿qué ratio esperaríamos?**

Con el ejemplo del caso:

```text
Zona          = Santiago
Hora          = 14:00
Lluvia        = 7.2 mm/hr
```

el modelo proyecta aproximadamente:

**Ratio ≈ 2.0**

Eso está por encima del umbral de saturación de `1.8`. El ratio proyectado es una **estimación estadística**, no una certeza.


# 2C. PASO 3: ¿Cómo calculamos el Earnings recomendado?

Se construyó un segundo modelo:

```text
RATIO ~ EARNINGS + PRECIPITATION_MM + HOUR + ZONE
```

Aquí sí se incluyó `EARNINGS`.

La lógica es:

1. El modelo 1 estima el **riesgo climático/operacional**.
2. El modelo 2 estima la relación histórica entre `EARNINGS` y `RATIO`.
3. Fijar un objetivo operacional: **RATIO = 1.8**, porque es el límite de saturación definido.


In [ ]:
# 2C. Modelo de calibración de earnings

earnings_model = smf.ols(
    "RATIO ~ EARNINGS + PRECIPITATION_MM + C(HOUR) + C(ZONE)",
    data=df
).fit()

earnings_coef = earnings_model.params["EARNINGS"]

print("Coeficiente histórico de EARNINGS:", round(earnings_coef, 5))
print("p-value:", f"{earnings_model.pvalues['EARNINGS']:.3e}")
print("R²:", round(earnings_model.rsquared, 3))

if earnings_coef >= 0:
    raise ValueError(
        "El coeficiente histórico de earnings no es negativo; "
        "no es válido usar este método de despeje."
    )


Coeficiente histórico de EARNINGS: -0.02678
p-value: 0.000e+00
R²: 0.834


In [ ]:
# 2C. Función que despeja el earnings necesario

TARGET_RATIO = 1.8

def recommended_earnings(
    zone,
    hour,
    precipitation,
    current_earnings,
    target_ratio=TARGET_RATIO
):
    # Crear una observación con EARNINGS = 0.
    # El modelo lineal permite despejar posteriormente EARNINGS.
    x0 = pd.DataFrame({
        "EARNINGS": [0],
        "PRECIPITATION_MM": [precipitation],
        "HOUR": [int(hour)],
        "ZONE": [zone]
    })

    intercept_at_conditions = float(
        earnings_model.predict(x0).iloc[0]
    )

    # Ratio = intercept_at_conditions + beta_earnings * EARNINGS
    # target = intercept + beta * earnings
    # earnings = (target - intercept) / beta
    required = (
        target_ratio - intercept_at_conditions
    ) / earnings_coef

    # No pagar menos que el earnings actual.
    recommended = max(current_earnings, required)

    recommended = int(np.ceil(recommended))

    return {
        "earnings_required_model": required,
        "earnings_recommended": recommended,
        "increment_mxn": recommended - current_earnings,
        "increment_pct": (
            (recommended / current_earnings - 1) * 100
            if current_earnings > 0 else np.nan
        )
    }

example_earnings = recommended_earnings(
    zone="Santiago",
    hour=14,
    precipitation=7.2,
    current_earnings=55
)

print(example_earnings)


{'earnings_required_model': np.float64(77.49982439652005), 'earnings_recommended': 78, 'increment_mxn': 23, 'increment_pct': 41.81818181818182}


###  Resultado

 El primer modelo estima el riesgo operacional a partir de lluvia, hora y zona. Después, un segundo modelo calibra el nivel de earnings necesario para llevar el ratio proyectado al umbral objetivo de 1.8. Para el ejemplo de Santiago, el cálculo produce aproximadamente 78 MXN frente a 55 MXN actuales.


# 2D. ¿El umbral de lluvia debe ser igual para todas las zonas?


**1 mm/hr** como threshold global porque el Módulo 1 mostró un cambio operacional importante alrededor de ese nivel

Después incorporar **sensibilidad por zona**:

- zonas de alta sensibilidad → mayor prioridad;
- zonas de sensibilidad media → riesgo intermedio;
- zonas de baja sensibilidad → no alertar únicamente por lluvia.


In [ ]:
# 2D. Clasificación de sensibilidad por zona

# P3 ya creó zone_sensitivity.

zone_sensitivity["sensitivity_group"] = pd.qcut(
    zone_sensitivity["rain_slope"],
    q=3,
    labels=["BAJA", "MEDIA", "ALTA"],
    duplicates="drop"
)

display(zone_sensitivity.round(4))


,ZONE,rain_slope,p_value,r_squared,sensitivity_group
13,Santiago,0.1915,0.0,0.8013,ALTA
1,Carretera Nacional,0.1339,0.0,0.8456,ALTA
12,Santa Catarina,0.1026,0.0,0.6981,ALTA
7,MTY_Apodaca_Huinalá,0.0847,0.0,0.8356,ALTA
5,Independencia,0.0748,0.0,0.7033,ALTA
11,San Pedro,0.0670,0.0,0.7680,MEDIA
4,Escobedo,0.0668,0.0,0.6460,MEDIA
0,Apodaca Centro,0.0658,0.0,0.7448,MEDIA
2,Centro,0.0610,0.0,0.7611,MEDIA
6,La Fe,0.0590,0.0,0.6260,BAJA


# 2E. Evitar alertas duplicadas

Definiremos un evento como:

```text
misma zona
+
misma ventana de riesgo
```

y se aplica un **cooldown de 2 horas**.

Si la condición sigue activa durante esas dos horas, no enviamos una alerta nueva.

Si la lluvia/riesgo desaparece y luego vuelve, se genera un nuevo evento.


In [ ]:
# 2E. Deduplicación de eventos

ALERT_COOLDOWN_MINUTES = 120

def build_event_id(zone, timestamp):
    timestamp = pd.Timestamp(timestamp)

    # Agrupar por bloques de 2 horas.
    block = (
        timestamp.floor("h")
        - pd.Timedelta(
            minutes=timestamp.minute % (ALERT_COOLDOWN_MINUTES)
        )
    )

    return f"{zone}_{block.strftime('%Y%m%d_%H%M')}"


def remove_duplicate_alerts(alerts):
    if alerts.empty:
        return alerts.copy()

    alerts = alerts.sort_values(
        ["ZONE", "FORECAST_TIME"]
    ).copy()

    alerts["EVENT_ID"] = alerts.apply(
        lambda r: build_event_id(
            r["ZONE"],
            r["FORECAST_TIME"]
        ),
        axis=1
    )

    return alerts.drop_duplicates(
        subset=["EVENT_ID"],
        keep="first"
    )


# 2F. Motor completo

Ahora juntamos todo:

```text
Forecast
   ↓
Precipitación + zona + hora
   ↓
Ratio proyectado
   ↓
Riesgo
   ↓
Earnings recomendado
   ↓
Deduplicación
   ↓
Alerta
```


In [ ]:
# 2F. Motor completo

def classify_risk(ratio_projected):
    if ratio_projected > 1.8:
        return "CRÍTICO"
    elif ratio_projected >= 1.5:
        return "ALTO"
    elif ratio_projected >= 1.2:
        return "MEDIO"
    else:
        return "BAJO"


def run_alert_engine(
    forecast_df,
    current_earnings_by_zone=None,
    lookahead_hours=2,
    rain_threshold=1.0
):
    if current_earnings_by_zone is None:
        current_earnings_by_zone = (
            df.groupby("ZONE")["EARNINGS"]
              .last()
              .to_dict()
        )

    data = forecast_df.copy()
    data = data.sort_values(["ZONE", "FORECAST_TIME"])


    if lookahead_hours is not None and not data.empty:
        start_time = data["FORECAST_TIME"].min()
        data = data[
            data["FORECAST_TIME"] <= start_time + pd.Timedelta(hours=lookahead_hours)
        ].copy()

    # Para cada zona/hora, usamos la precipitación forecast
    # El ratio se proyecta directamente con el modelo histórico
    data["HOUR"] = data["FORECAST_TIME"].dt.hour

    data["RATIO_PROJECTED"] = data.apply(
        lambda r: project_ratio(
            r["ZONE"],
            r["HOUR"],
            r["PRECIPITATION_FORECAST_MM"]
        ),
        axis=1
    )

    data["RISK"] = data["RATIO_PROJECTED"].apply(classify_risk)

    data["CURRENT_EARNINGS"] = data["ZONE"].map(
        current_earnings_by_zone
    )

    # Solo generamos alerta cuando:
    # 1) el ratio proyectado cruza saturación, o
    # 2) lluvia > 1 mm/hr + ratio ya elevado.
    data["TRIGGER"] = (
        (data["RATIO_PROJECTED"] > 1.8) |
        (
            (data["PRECIPITATION_FORECAST_MM"] > rain_threshold) &
            (data["RATIO_PROJECTED"] >= 1.5)
        )
    )

    alerts = data[data["TRIGGER"]].copy()

    if alerts.empty:
        return alerts

    recs = alerts.apply(
        lambda r: recommended_earnings(
            zone=r["ZONE"],
            hour=r["HOUR"],
            precipitation=r["PRECIPITATION_FORECAST_MM"],
            current_earnings=r["CURRENT_EARNINGS"]
        ),
        axis=1,
        result_type="expand"
    )

    alerts = pd.concat(
        [alerts.reset_index(drop=True), recs],
        axis=1
    )

    alerts = remove_duplicate_alerts(alerts)

    risk_order = {
        "CRÍTICO": 4,
        "ALTO": 3,
        "MEDIO": 2,
        "BAJO": 1
    }

    alerts["RISK_ORDER"] = alerts["RISK"].map(risk_order)

    return alerts[
        [
            "ZONE",
            "FORECAST_TIME",
            "PRECIPITATION_FORECAST_MM",
            "RATIO_PROJECTED",
            "RISK",
            "CURRENT_EARNINGS",
            "earnings_recommended",
            "increment_mxn",
            "increment_pct",
            "EVENT_ID",
            "MAPPING_METHOD",
            "RISK_ORDER"
        ]
    ].sort_values(
        ["RISK_ORDER", "RATIO_PROJECTED"],
        ascending=[False, False]
    ).drop(columns=["RISK_ORDER"])


alerts = run_alert_engine(forecast)

print("Alertas generadas:", len(alerts))
display(alerts.head(20).round(2))


Alertas generadas: 0


,ZONE,LAT_QUERY,LON_QUERY,FORECAST_TIME,PRECIPITATION_FORECAST_MM,MAPPING_METHOD,API_LATITUDE,API_LONGITUDE,HOUR,RATIO_PROJECTED,RISK,CURRENT_EARNINGS,TRIGGER



Cada fila representa una oportunidad de acción.

Ejemplo:

```text
Zona: Santiago
Forecast: 7.2 mm/hr
Ratio proyectado: ~2.0
Riesgo: CRÍTICO
Earnings actual: $55
Earnings recomendado: ~$78
```

Entonces Operations recibe una recomendación concreta

### El sistema responde las 4 preguntas del caso

| Pregunta | Respuesta |
|---|---|
| ¿Qué lluvia dispara alerta? | >1 mm/hr como trigger preventivo, junto con riesgo operacional |
| ¿Es igual para todas las zonas? | El trigger base es global, pero la prioridad se ajusta por sensibilidad |
| ¿Cuánto subir earnings? | Se despeja desde el modelo histórico; ejemplo $55 → ~$78 |
| ¿Cómo evitar duplicados? | `EVENT_ID` + cooldown de 2 horas |




# 2G. Ejemplo reproducible del caso

Voya reproducir explícitamente el ejemplo solicitado por Rappi:

```text
Zona: Santiago
Precipitación: 7.2 mm/hr
Earnings actual: 55 MXN
```

El objetivo es demostrar que nuestro motor puede llegar a un earnings recomendado cercano a **78 MXN** a partir del histórico.


In [ ]:
# 2G — Reproducir el ejemplo del caso

demo = recommended_earnings(
    zone="Santiago",
    hour=14,
    precipitation=7.2,
    current_earnings=55
)

demo_ratio = project_ratio(
    zone="Santiago",
    hour=14,
    precipitation=7.2
)

print("=" * 70)
print("DEMO — EJEMPLO DEL CASO")
print("=" * 70)
print("Zona: Santiago")
print("Hora: 14:00")
print("Precipitación esperada: 7.2 mm/hr")
print(f"Ratio proyectado antes del incentivo: {demo_ratio:.2f}")
print("Earnings actual: $55 MXN")
print(f"Earnings recomendado: ${demo['earnings_recommended']} MXN")
print(f"Incremento: ${demo['increment_mxn']:.0f} MXN")
print(f"Incremento porcentual: {demo['increment_pct']:.1f}%")


DEMO — EJEMPLO DEL CASO
Zona: Santiago
Hora: 14:00
Precipitación esperada: 7.2 mm/hr
Ratio proyectado antes del incentivo: 2.01
Earnings actual: $55 MXN
Earnings recomendado: $78 MXN
Incremento: $23 MXN
Incremento porcentual: 41.8%


# Resumen - Módulo 2

## Motor de decisión

```text
FORECAST
   ↓
ZONA + HORA + LLUVIA
   ↓
RATIO PROYECTADO
   ↓
RIESGO
   ↓
EARNINGS RECOMENDADO
   ↓
DEDUPLICACIÓN
   ↓
ALERTA
```

### Reglas principales

| Saturación | `RATIO > 1.8` |
| Trigger climático | `precipitación > 1 mm/hr` combinado con presión operacional |
| Anticipación MVP | 2 horas |
| Earnings recomendado | Se calcula con el modelo histórico y objetivo `RATIO = 1.8` |
| Duplicados | `EVENT_ID` + cooldown de 2 horas |
| Sensibilidad | Se considera la respuesta histórica de cada zona |

